In [264]:
%pip install nba_api pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [265]:
#load data into csv and data folder

from pathlib import Path
import time
from nba_api.stats.endpoints import leaguegamelog

data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

seasons = ["2021-22", "2022-23", "2023-24", "2024-25"]

for season in seasons:
    file_path = data_folder / f"player_games_{season}.csv"

    # Reuse files you have already downloaded.
    if file_path.exists():
        print(f"Already saved: {file_path.name}")
        continue

    print(f"Downloading {season}...")

    try:
        response = leaguegamelog.LeagueGameLog(
            season=season,
            season_type_all_star="Regular Season",
            player_or_team_abbreviation="P",
            timeout=60
        )

        season_df = response.get_data_frames()[0]

        if season_df.empty:
            print(f"No rows returned for {season}; file not saved.")
        else:
            season_df.to_csv(file_path, index=False)
            print(f"Saved {len(season_df):,} rows to {file_path.name}")

    except Exception as error:
        print(f"Failed for {season}: {error}")

    # Space out requests to the server.
    time.sleep(3)

print("Data folder:", data_folder.resolve())

Already saved: player_games_2021-22.csv
Already saved: player_games_2022-23.csv
Already saved: player_games_2023-24.csv
Already saved: player_games_2024-25.csv
Data folder: /Users/tylerphan/Desktop/NBA ML/data


In [ ]:
import pandas as pd
import numpy as np
#load first season



def create_features(games):
    df = games.copy()

    df["PTS_AVG_5"] = (
    df.groupby("PLAYER_ID")["PTS"].transform(lambda points: points.shift(1).rolling(5).mean())

    )

    df["PTS_SEASON_TO_DATE"] = (
        df.groupby("PLAYER_ID")["PTS"]
        .transform(lambda points: points.shift(1).expanding().mean())
    )

    df["AVG_FGA"] = (
        df.groupby("PLAYER_ID")["FGA"].transform(lambda fga: fga.shift(1).expanding().mean())
    )


    df["AVG_FG3A"] = (
        df.groupby("PLAYER_ID")["FG3A"].transform(lambda fg3a: fg3a.shift(1).expanding().mean())
    )

    df["AVG_FG3A_LAST_5"] = (
        df.groupby("PLAYER_ID")["FG3A"].transform(lambda fg3a: fg3a.shift(1).rolling(5).mean())
    )

    df["AVG_FTA_LAST_5"] = (
        df.groupby("PLAYER_ID")["FTA"].transform(lambda fta: fta.shift(1).rolling(5).mean())
    )
   

    df["IS_HOME"] = df["MATCHUP"].str.contains("vs")

    df["FG2A"] = df["FGA"] - df["FG3A"]
    df["AVG_FG2A_LAST_5"] = (
        df.groupby("PLAYER_ID")["FG2A"]
        .transform(lambda attempts: attempts.shift(1).rolling(5).mean())
    )
    
    by_player = df.groupby("PLAYER_ID")

    prior_3pm = by_player["FG3M"].transform(
        lambda s: s.shift(1).cumsum()
    )
    prior_3pa = by_player["FG3A"].transform(
        lambda s: s.shift(1).cumsum()
    )
    df["AVG_FG3P"] = prior_3pm / prior_3pa.where(prior_3pa > 0)

    df["FG2M"] = df["FGM"] - df["FG3M"]
    prior_2pm = by_player["FG2M"].transform(lambda s: s.shift(1).cumsum())
    prior_2pa = by_player["FG2A"].transform(lambda s: s.shift(1).cumsum())

    df["AVG_FG2P"] = prior_2pm / prior_2pa.where(prior_2pa > 0)

    prior_ftm = by_player["FTM"].transform(
    lambda s: s.shift(1).cumsum()
)
    prior_fta = by_player["FTA"].transform(
        lambda s: s.shift(1).cumsum()
    )
    df["AVG_FTP"] = prior_ftm / prior_fta.where(prior_fta > 0)

    df["AVG_MIN"] = (
        df.groupby("PLAYER_ID")["MIN"].transform(lambda min: min.shift(1).expanding().mean())
    )

    #defense calculations

    team_games = (
        df.groupby(["GAME_ID", "GAME_DATE", "TEAM_ID","TEAM_ABBREVIATION"])["PTS"]
        .sum()
        .reset_index(name="TEAM_PTS")
    )
    team_games["GAME_DATE"] = pd.to_datetime(team_games["GAME_DATE"])
    team_games = team_games.sort_values(["GAME_DATE", "GAME_ID"])


    game_total = team_games.groupby("GAME_ID")["TEAM_PTS"].transform("sum")

    team_games["PTS_ALLOWED"] = game_total - team_games["TEAM_PTS"]

    team_games["PTS_ALLOWED_LAST_10"] = (
        team_games.groupby("TEAM_ID")["PTS_ALLOWED"].transform(lambda points: points.shift(1).rolling(10).mean())
    )

    df["OPPONENT"] = df["MATCHUP"].str.split().str[-1]

    defense_lookup = team_games[["GAME_ID", "TEAM_ABBREVIATION", "PTS_ALLOWED_LAST_10"]].rename(columns={
        "TEAM_ABBREVIATION": "OPPONENT",
        "PTS_ALLOWED_LAST_10": "OPP_PTS_ALLOWED_LAST_10"
    })

    df = df.merge(
        defense_lookup,
        on=["GAME_ID", "OPPONENT"],
        how="left",
        validate= "many_to_one"
    )
    
    df["AVG_MIN_LAST_3"] = (
    df.groupby("PLAYER_ID")["MIN"]
    .transform(lambda s: s.shift(1).rolling(3).mean())
    )    

    
    return df


training_seasons = []

for season in ["2021-22", "2022-23"]:
    raw = pd.read_csv(
        f"data/player_games_{season}.csv",
        dtype={"GAME_ID": str, "SEASON_ID": str}
    )
    training_seasons.append(create_features(raw))

df = pd.concat(training_seasons, ignore_index=True)




/var/folders/l3/vh1wq94d3j5_m4xcwnbz2zmr0000gn/T/ipykernel_8134/246044768.py:7: DtypeWarning: Columns (0: SEASON_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"data/player_games_{season}.csv", dtype={"GAME_ID": str})


In [267]:
# training
from sklearn.linear_model import LinearRegression


features = ["PTS_AVG_5", 
            "IS_HOME",
            "AVG_FTA_LAST_5",
            "AVG_FG2A_LAST_5",
            "AVG_FG3A_LAST_5",
            "PTS_SEASON_TO_DATE",
            "AVG_FG2P",
            # "AVG_FG3P",
            # "AVG_FTP",
            "AVG_MIN",
            "AVG_MIN_LAST_3",
            "OPP_PTS_ALLOWED_LAST_10",
            
            ]

df = create_features(df)

train = df.dropna(subset=features + ["PTS"])
X_train = train[features]
Y_train = train["PTS"]

model = LinearRegression()
model.fit(X_train, Y_train)

print(pd.Series(model.coef_, index=X_train.columns))

print(model.intercept_)

PTS_AVG_5                  0.000933
IS_HOME                   -0.003178
AVG_FTA_LAST_5             0.247204
AVG_FG2A_LAST_5            0.333251
AVG_FG3A_LAST_5            0.286545
PTS_SEASON_TO_DATE         0.698245
AVG_FG2P                  -0.260286
AVG_MIN                   -0.142086
AVG_MIN_LAST_3             0.125739
OPP_PTS_ALLOWED_LAST_10    0.056688
dtype: float64
-5.2936619722492395


In [268]:
#validation
from sklearn.metrics import mean_absolute_error

validation_raw = pd.read_csv("data/player_games_2023-24.csv", dtype={"GAME_ID":str})

validation = create_features(validation_raw)



val = validation.dropna(subset=features + ["PTS"]).copy()

X_val = val[features]
Y_val = val["PTS"]


predictions = model.predict(X_val)

val["PREDICTED_PTS"] = predictions
val["ABS_ERROR"] = (val["PTS"] - val["PREDICTED_PTS"]).abs()

print("Model MAE:", mean_absolute_error(Y_val, predictions))
print("Baseline MAE:", mean_absolute_error(Y_val, val["PTS_AVG_5"]))
print("Player-games evaluated:", len(val))

print(
    val[["PLAYER_NAME", "GAME_DATE", "PTS",
         "PREDICTED_PTS", "PTS_AVG_5", "ABS_ERROR"]]
    .sample(100, random_state=42)
    .round(2)
    .to_string(index=False)
)

val["SIGNED_ERROR"] = val["PREDICTED_PTS"] - val["PTS"]
print("Mean signed error:", val["SIGNED_ERROR"].mean())

high_scorers = val[val["PTS_SEASON_TO_DATE"] >= 20]

print("Player-games:", len(high_scorers))
print("Model MAE:", mean_absolute_error(
    high_scorers["PTS"], high_scorers["PREDICTED_PTS"]
))
print("Baseline MAE:", mean_absolute_error(
    high_scorers["PTS"], high_scorers["PTS_AVG_5"]
))


predicted_direction = np.sign(val["PREDICTED_PTS"] - val["PTS_AVG_5"])
actual_direction = np.sign(val["PTS"] - val["PTS_AVG_5"])

val["DIRECTION_CORRECT"] = predicted_direction == actual_direction

print(f"Direction accuracy: {val['DIRECTION_CORRECT'].mean():.1%}")

val["MIN_CHANGE"] = val["MIN"] - val["AVG_MIN_LAST_5"]

val["MIN_CHANGE_GROUP"] = pd.cut(
    val["MIN_CHANGE"].abs(),
    bins=[0, 5, 10, float("inf")],
    labels=["Within 5 minutes", "5–10 minutes", "Over 10 minutes"],
    include_lowest=True
)

print(
    val.groupby("MIN_CHANGE_GROUP", observed=True)
    .agg(
        GAMES=("ABS_ERROR", "size"),
        MAE=("ABS_ERROR", "mean")
    )
    .round(2)
)

Model MAE: 4.616591924344057
Baseline MAE: 4.772917319486101
Player-games evaluated: 22339
             PLAYER_NAME  GAME_DATE  PTS  PREDICTED_PTS  PTS_AVG_5  ABS_ERROR
           Chet Holmgren 2024-03-04   15          18.34       21.8       3.34
         Kelly Oubre Jr. 2024-03-16   22          17.82       18.2       4.18
          Paolo Banchero 2024-02-04   20          25.64       27.4       5.64
          Onyeka Okongwu 2024-02-05   18          10.54       13.2       7.46
         Khris Middleton 2024-03-24   11          14.67       16.8       3.67
         Jordan Clarkson 2023-11-19   22          21.82       26.6       0.18
              Saddiq Bey 2024-01-22   13          14.07        9.4       1.07
             Cory Joseph 2024-01-07    0           2.33        1.4       2.33
            Terance Mann 2023-12-23    0           7.53        6.2       7.53
            Zach Collins 2024-03-23   11           9.94        8.0       1.06
             Josh Okogie 2023-12-02    3           